# SatMesh — Track A Training
DeepGlobe Road Extraction · U-Net ResNet34 · clDice loss

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'albumentations>=2.0', 'segmentation-models-pytorch', 'timm'])

import torch
print(f'deps OK  |  torch {torch.__version__}  |  CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}  (capability {cap})')
    assert cap >= (7, 0), (
        f'GPU capability {cap} is too old for this PyTorch build. '
        'In the Kaggle editor: Settings -> Accelerator -> "GPU T4 x2", then Save & Run All.'
    )
    _t = torch.randn(8, 8, device='cuda'); _ = (_t @ _t).sum().item()
    print('CUDA kernel launch OK')

In [ ]:
import json, os, glob, random
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

# ── diagnose dataset structure ────────────────────────────────────────────────
print('\n/kaggle/input layout:')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root.replace('/kaggle/input', '').count(os.sep)
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth >= 2 and files:
        print(f'{indent}  files ({len(files)} total): {files[:6]}')
        break

# ── robust pair discovery (handles _sat.jpg/_mask.png or .jpg/.png) ──────────
pairs = []
for root, dirs, files in os.walk('/kaggle/input'):
    sat = [f for f in files if f.endswith('_sat.jpg')]
    if sat:
        for sf in sat:
            sp = os.path.join(root, sf)
            mp = sp.replace('_sat.jpg', '_mask.png')
            if os.path.exists(mp):
                pairs.append((sp, mp))
        if pairs:
            print(f'\nPairs found using _sat.jpg/_mask.png convention in:\n  {root}')
            break

if not pairs:
    for root, dirs, files in os.walk('/kaggle/input'):
        jpgs = [f for f in files if f.lower().endswith('.jpg')]
        for jf in jpgs:
            sp = os.path.join(root, jf)
            mp = os.path.join(root, os.path.splitext(jf)[0] + '.png')
            if os.path.exists(mp):
                pairs.append((sp, mp))
        if pairs:
            print(f'\nPairs found using .jpg/.png convention in:\n  {root}')
            break

assert len(pairs) > 0, 'No image/mask pairs found — see /kaggle/input layout above'
print(f'Total pairs: {len(pairs)}  |  sample: {pairs[0]}')

SUBSET = 3000
random.shuffle(pairs)
pairs = pairs[:SUBSET]
n = len(pairs)
train_pairs = pairs[:int(0.8*n)]
val_pairs   = pairs[int(0.8*n):int(0.9*n)]
test_pairs  = pairs[int(0.9*n):]
print(f'train {len(train_pairs)}  val {len(val_pairs)}  test {len(test_pairs)}')

In [ ]:
IMG   = 512
BATCH = 4

train_tf = A.Compose([
    A.Resize(IMG, IMG),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3), A.RandomRotate90(p=0.5),
    A.Affine(scale=(0.9,1.1), translate_percent=0.1, rotate=(-15,15), p=0.4),
    A.RandomShadow(p=0.4),
    A.CoarseDropout(num_holes_range=(2,12), hole_height_range=(8,40),
                    hole_width_range=(8,40), fill=0, p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.RandomGamma(gamma_limit=(80,120), p=0.3),
    A.GaussNoise(std_range=(0.01,0.05), p=0.3),
    A.Normalize(), ToTensorV2(),
])
val_tf = A.Compose([A.Resize(IMG, IMG), A.Normalize(), ToTensorV2()])

class RoadDS(Dataset):
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        sp, mp = self.items[i]
        img = cv2.cvtColor(cv2.imread(sp), cv2.COLOR_BGR2RGB)
        msk = (cv2.imread(mp, cv2.IMREAD_GRAYSCALE) > 127).astype('float32')
        a = self.tf(image=img, mask=msk)
        return a['image'], a['mask'].unsqueeze(0)

pin = (DEVICE == 'cuda')
train_dl = DataLoader(RoadDS(train_pairs, train_tf), batch_size=BATCH,
                      shuffle=True, num_workers=2, pin_memory=pin)
val_dl   = DataLoader(RoadDS(val_pairs,   val_tf),   batch_size=BATCH,
                      shuffle=False, num_workers=2, pin_memory=pin)
print('Dataloaders ready')

In [ ]:
# ── clDice loss ───────────────────────────────────────────────────────────────
def _soft_erode(x):  return -F.max_pool2d(-x, 3, 1, 1)
def _soft_dilate(x): return  F.max_pool2d( x, 3, 1, 1)

def soft_skel(x, iters=5):
    skel = F.relu(x - _soft_dilate(_soft_erode(x)))
    for _ in range(iters-1):
        x    = _soft_erode(x)
        d    = F.relu(x - _soft_dilate(_soft_erode(x)))
        skel = skel + F.relu(d - skel*d)
    return skel

def cl_dice_loss(logits, target, iters=5, smooth=1.0):
    pred = torch.sigmoid(logits)
    sp, st = soft_skel(pred, iters), soft_skel(target, iters)
    tp = ((sp*target).sum()+smooth) / (sp.sum()+smooth)
    ts = ((st*pred).sum()+smooth)   / (st.sum()+smooth)
    return 1.0 - 2.0*tp*ts / (tp+ts)

model     = smp.Unet('resnet34', encoder_weights='imagenet', in_channels=3, classes=1).to(DEVICE)
dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss  = nn.BCEWithLogitsLoss()
ALPHA, BETA, GAMMA = 0.4, 0.3, 0.3

EPOCHS = 20
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt, max_lr=1e-3, steps_per_epoch=len(train_dl), epochs=EPOCHS)

def batch_metrics(logits, target, thr=0.5):
    pred  = (torch.sigmoid(logits) > thr).float()
    inter = (pred*target).sum((1,2,3))
    union = ((pred+target)>0).float().sum((1,2,3))
    iou  = ((inter+1e-6)/(union+1e-6)).mean().item()
    dice = ((2*inter+1e-6)/(pred.sum((1,2,3))+target.sum((1,2,3))+1e-6)).mean().item()
    return iou, dice

print('Model ready')

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
out_dir   = '/kaggle/working'
best_path = os.path.join(out_dir, 'best_model.pth')
meta_path = os.path.join(out_dir, 'best_model_meta.json')
test_path = os.path.join(out_dir, 'test_pairs.json')
best_iou  = 0.0
epoch_log = []   # will be plotted in the next cell

with open(test_path, 'w') as f: json.dump(test_pairs, f)

for ep in range(1, EPOCHS+1):
    model.train(); train_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        out  = model(x)
        loss = ALPHA*dice_loss(out,y) + BETA*bce_loss(out,y) + GAMMA*cl_dice_loss(out,y)
        loss.backward(); opt.step(); scheduler.step()
        train_loss += loss.item()

    model.eval(); all_iou, all_dice = [], []
    with torch.no_grad():
        for x, y in val_dl:
            iou, dice = batch_metrics(model(x.to(DEVICE)), y.to(DEVICE))
            all_iou.append(iou); all_dice.append(dice)

    avg_loss = train_loss / len(train_dl)
    mean_iou  = float(np.mean(all_iou))
    mean_dice = float(np.mean(all_dice))
    lr_now    = scheduler.get_last_lr()[0]

    epoch_log.append({'epoch': ep, 'loss': avg_loss,
                      'val_iou': mean_iou, 'val_dice': mean_dice})

    print(f'Ep {ep:2d}/{EPOCHS}  loss={avg_loss:.4f}  '
          f'val_iou={mean_iou:.4f}  val_dice={mean_dice:.4f}  lr={lr_now:.2e}')

    if mean_iou > best_iou:
        best_iou = mean_iou
        torch.save(model.state_dict(), best_path)
        with open(meta_path, 'w') as f:
            json.dump({'best_val_iou': round(best_iou,4), 'epoch': ep,
                       'img_size': IMG, 'subset': SUBSET,
                       'loss_weights': {'dice':ALPHA,'bce':BETA,'cldice':GAMMA},
                       'device': DEVICE}, f, indent=2)
        print(f'  >> new best saved')

# save epoch log for plotting
with open(os.path.join(out_dir, 'epoch_metrics.json'), 'w') as f:
    json.dump(epoch_log, f, indent=2)
print(f'\nTraining done. Best val IoU: {best_iou:.4f}')

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
eps       = [r['epoch']    for r in epoch_log]
losses    = [r['loss']     for r in epoch_log]
val_ious  = [r['val_iou']  for r in epoch_log]
val_dices = [r['val_dice'] for r in epoch_log]

best_ep = eps[val_ious.index(max(val_ious))]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(eps, losses, 'o-', color='#d62728', lw=2, ms=4)
axes[0].set_title('Training Loss  (0.4·Dice + 0.3·BCE + 0.3·clDice)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(eps, val_ious, 's-', color='#1f77b4', lw=2, ms=4)
axes[1].axvline(best_ep, color='#2ca02c', ls='--', alpha=0.8,
                label=f'Best: {max(val_ious):.4f} @ ep {best_ep}')
axes[1].set_title('Validation IoU')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('IoU')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(eps, val_dices, '^-', color='#9467bd', lw=2, ms=4)
axes[2].set_title('Validation Dice')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Dice')
axes[2].grid(True, alpha=0.3)

fig.suptitle(f'SatMesh Track A — DeepGlobe  (best val IoU {max(val_ious):.4f})', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sample predictions on validation images ───────────────────────────────────
model.load_state_dict(torch.load(best_path, map_location=DEVICE, weights_only=True))
model.eval()

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

x_batch, y_batch = next(iter(val_dl))
with torch.no_grad():
    pred_batch = (torch.sigmoid(model(x_batch.to(DEVICE))) > 0.5).float().cpu()

N = min(4, x_batch.shape[0])
fig, axes = plt.subplots(N, 3, figsize=(12, 4*N))
if N == 1: axes = [axes]

for i in range(N):
    img_vis = np.clip(x_batch[i].permute(1,2,0).numpy() * STD + MEAN, 0, 1)
    axes[i][0].imshow(img_vis);              axes[i][0].set_title('Satellite'); axes[i][0].axis('off')
    axes[i][1].imshow(y_batch[i,0], cmap='gray');    axes[i][1].set_title('Ground truth'); axes[i][1].axis('off')
    axes[i][2].imshow(pred_batch[i,0], cmap='gray'); axes[i][2].set_title('Prediction'); axes[i][2].axis('off')

fig.suptitle(f'SatMesh — Road Segmentation  (val IoU {best_iou:.4f})', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(out_dir, 'sample_predictions.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
meta = json.load(open(meta_path))
print('=' * 50)
print('SatMesh Track A — Run Summary')
print('=' * 50)
for k, v in meta.items(): print(f'  {k:<25} {v}')
print(f'  {"final_val_iou":<25} {val_ious[-1]:.4f}')
print(f'  {"final_val_dice":<25} {val_dices[-1]:.4f}')
print('=' * 50)
print('\nOutputs in /kaggle/working:')
for fname in ['best_model.pth','best_model_meta.json','test_pairs.json',
              'epoch_metrics.json','training_curves.png','sample_predictions.png']:
    path = os.path.join(out_dir, fname)
    size = os.path.getsize(path) if os.path.exists(path) else 0
    print(f'  {fname:<35} {size/1024:.1f} KB')
print('\n-- Review plots above. Give green light to download best_model.pth --')